<a href="https://colab.research.google.com/github/alemjarebica-cloud/ML-flyrank-AlemJ/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alemjarebica-cloud/ML-flyrank-AlemJ/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is ranking signal analysis, the goal is not just to say "yes/no, this is declining" for one piece of content, it is to order all content by priority for refresh, so an editor knows what to fix first. That means the model's output is not a hard category but a score (probability of decline), which is then used to rank the whole content list.
It is not clustering, because I am not looking for groups of similar content without a target. I have a clear, measurable outcome (is_declining_label). It is not pure classification either, because the end deliverable is not a "yes/no" call, it's a ranked list. But under the hood I use a classification model (predicting probability of decline), whose output I then sort, so this is scoring/ranking built on top of classification.
Action this supports: The ranked output feeds directly into an editor's weekly refresh queue, instead of guessing which of thousands of articles to check, they get a shortlist of the top-K highest-risk items to review first.

In [34]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")
df.head()

Loaded 30000 rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(df["trend_direction"].value_counts())
print(f"\nCategories found: {df['trend_direction'].nunique()}")
print(f"Rows with a usable trend signal: {(df['trend_direction'] != 'new').sum()} / {len(df)}")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Categories found: 5
Rows with a usable trend signal: 27764 / 30000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The dataset does not come with a ready-made decline label, so I build one from trend_direction, which reflects an outcome that was actually observed over the trailing window, not something I defined by a rule of my own.
trend_direction has 5 categories: down, stable, up, flat, and new. I drop new (2,236 rows) before building the label, because those items are too recent to have a trend at all, labeling them as "declining" or "not declining" would just be a guess, not an observation.
After dropping new, the label is:
is_declining_label = 1 if trend_direction == "down", else 0 (covering stable, up, and flat as the negative class).
This leaves 27,764 labeled rows, split roughly 59% declining / 41% not declining, close enough to balanced that I do not need to worry about severe class imbalance later.
Leakage warning: since this label comes directly from trend_direction (and indirectly from trend_pct), neither of those two columns can ever be used as a model feature. They are the answer, not a clue.

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df_labeled = df[df["trend_direction"] != "new"].copy()
df_labeled["is_declining_label"] = (df_labeled["trend_direction"] == "down").astype(int)

print(f"Rows: {len(df_labeled)}")
print(df_labeled["is_declining_label"].value_counts(normalize=True))

Rows: 27764
is_declining_label
1    0.585723
0    0.414277
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: precision@K
Since the real deliverable is a ranked priority list (not a single yes/no call), the metric has to match how the output gets used: an editor only has time to fix a handful of items, so what matters is whether the top of the list is actually correct, not overall accuracy across all 27,764 items.
precision@K = of the top K items the model ranks as highest-priority (highest predicted probability of decline), what fraction are actually declining (is_declining_label == 1)?
I will use K=100 as a realistic "editor's weekly queue" size. A trivial baseline (randomly picking 100 items) would get ~58.6% precision@100, since that is the base rate, so "good" means clearly beating that baseline, not just being above 50%.
I am not using plain accuracy, because it doesn't reflect how the output is consumed, nobody looks at all 27,764 predictions, they look at the top of the queue.

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
K = 100
base_rate = df_labeled["is_declining_label"].mean()
print(f"Baseline precision@{K} (random ordering) = {base_rate:.3f}")

Baseline precision@100 (random ordering) = 0.586


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one content item, for one client
Each row represents a single piece of content (an article or page) belonging to one client, with 90 days of trailing performance metrics attached to it. The content_id and client_id columns are pseudonyms used only for grouping and joining data, they are never used as model features.

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(f"Unique content items: {df_labeled['content_id'].nunique()}")
print(f"Unique clients: {df_labeled['client_id'].nunique()}")


dupes = df_labeled.duplicated(subset=["content_id"]).sum()
print(f"Duplicate content_id rows: {dupes}")

df_labeled[["content_id", "client_id", "content_type", "word_count",
            "ctr", "avg_position", "engagement_rate", "is_declining_label"]].sample(5, random_state=42)

Unique content items: 27764
Unique clients: 31
Duplicate content_id rows: 0


,content_id,client_id,content_type,word_count,ctr,avg_position,engagement_rate,is_declining_label
25174,content_3627b5dea537,client_3fdba35f04,keyword article,1547.0,0.12,9.9,0.0,1
6498,content_9d99f5f3f833,client_19581e27de,keyword article,NaN,0.00,11.3,0.0,1
16548,content_2ad23dedb3a9,client_6208ef0f77,keyword article,5171.0,0.36,16.4,0.0,0
18481,content_fe868b206060,client_4ec9599fc2,feedly article,1272.0,0.36,7.2,0.0,1
28218,content_d7e222df21c9,client_e629fa6598,keyword article,NaN,0.10,56.7,10.0,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML beats a fixed rule here:
A fixed rule (e.g. "if ctr < 0.2, flag as declining") would need to guess the right threshold for every signal separately, and ignore how signals interact. For example, low engagement_rate might mean nothing if avg_position is already excellent, but might be a strong warning sign if position is weak. The right combination of thresholds differs by content_type too, some types have almost no word_count data at all, so a rule built around word count would misfire on those.
A fixed rule also cannot weigh 30+ signals against each other or adjust which signals matter most, it can only check one condition at a time. ML can learn which combinations of signals, and in what proportions, actually correlate with real decline, something no single if-statement can capture.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
for col in ["ctr", "avg_position", "engagement_rate", "scroll_rate"]:
    print(f"{col}: mean when declining = {df_labeled[df_labeled['is_declining_label']==1][col].mean():.2f}, "
          f"mean when not = {df_labeled[df_labeled['is_declining_label']==0][col].mean():.2f}")

ctr: mean when declining = 0.32, mean when not = 0.62
avg_position: mean when declining = 15.94, mean when not = 18.17
engagement_rate: mean when declining = 2.44, mean when not = 2.75
scroll_rate: mean when declining = 18.13, mean when not = 16.08


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.